# LOG635 – Laboratoire 2 : création de l'EnsembleB
**Parties 1 à 4 :** préparation de l'EnsembleB, prétraitement (taille + ROI), altérations / augmentation, extraction des caractéristiques.

In [ ]:
# ABSOLUTE PATH DU SCRIPT SUR LE ROBOT : /chemin/absolu/vers/le/script/sur/le/robot.py   <-- À COMPLÉTER
from pathlib import Path
from collections import Counter
import random
import pickle
import shutil

import cv2
import imutils
import numpy as np
import matplotlib.pyplot as plt

---
## 0 | PARAMÈTRES
Tout ce qu'il faut modifier est dans la cellule suivante.

In [ ]:
# ----- Chemins -----
DATA_DIR = Path.cwd() / "EnsembleA"            # dossier EnsembleA (Cercles/, Triangles/, Hexagones/, Diamants/)
OUT_DIR  = Path.cwd() / "EnsembleB_pretraite"  # images prétraitées, classées par étiquette (à remettre dans le .zip)

# ----- Reproductibilité -----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# ----- Partie 1 : choix des symboles (>= 8) -----
SELECTED_LABELS = ["Cercle2", "Cercle5",
                   "Triangle2", "Triangle5",
                   "Hexagone2", "Hexagone5",
                   "Diamant2", "Diamant5"]
N_PER_SYMBOL = 10          # nombre d'images ORIGINALES retenues par symbole
EXCLUDE_FILES = set()      # ex. {"12_Diamant5.jpg"} pour écarter une image floue/mauvaise

# ----- Partie 2 : prétraitement -----
RESIZE_FACTOR = 0.5        # 0.5 => ex. 320x140 -> 160x70 (None pour désactiver)
ROI_SIZE = (64, 64)        # (largeur, hauteur) FINALE de la ROI : identique pour toutes les images
ROI_MARGIN = 0.08          # marge ajoutée autour de la boîte englobante (8 %)
MIN_ROI_AREA_RATIO = 0.02  # contour plus petit que 2 % de l'image => ROI jugée invalide (repli sur l'image entière)

# ----- Partie 3 : altérations -----
APPLY_BLUR = False         # flou gaussien appliqué à TOUTES les images (optionnel)
BLUR_KERNEL = (3, 3)
N_AUG_PER_IMAGE = 20       # images générées par image originale (au moins 100 au total exigé)
USE_FLIPS = False          # retournements : désactivés par défaut (le robot ne verra pas de marqueurs en miroir)

# ----- Partie 4 : caractéristiques -----
USE_EXTRA_FEATURES = True  # ajoute : nombre de formes + excentricité de la plus grande forme
MAX_SHAPES = 10            # constante de normalisation du nombre de formes

In [ ]:
def show_grid(images, titles=None, ncols=8, size=2.0):
    """Affiche une grille d'images en niveaux de gris."""
    n = len(images)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for i, (ax, im) in enumerate(zip(axes, images)):
        ax.imshow(im, cmap="gray", vmin=0, vmax=255)
        if titles is not None:
            ax.set_title(titles[i], fontsize=8)
    plt.tight_layout()
    plt.show()

---
## 1 | PRÉPARATION DE L'ENSEMBLEB
Sélection d'au moins 8 symboles dans l'EnsembleA, avec `N_PER_SYMBOL` images par symbole.

**Justification du choix des symboles** *(à compléter pour le rapport)* :
> - 4 formes différentes (cercle, triangle, hexagone, diamant) × 2 valeurs (2 et 5) : ...
> - Pourquoi 2 et 5 : ...
> - Compromis nombre de classes / difficulté : ...

In [ ]:
def natural_key(path):
    """Trie '2_Diamant5.jpg' avant '10_Diamant5.jpg'."""
    digits = "".join(ch for ch in path.stem.split("_")[0] if ch.isdigit())
    return int(digits) if digits else 0


def collect_images(label):
    """Toutes les images d'un symbole, quel que soit son dossier parent (Triangles/, Diamants/, ...)."""
    folders = [d for d in DATA_DIR.glob(f"*/{label}") if d.is_dir()]
    if not folders:
        raise FileNotFoundError(f"Aucun dossier '{label}' trouvé sous {DATA_DIR}")
    files = [f for d in folders for f in d.iterdir()
             if f.suffix.lower() in {".jpg", ".jpeg", ".png"} and f.name not in EXCLUDE_FILES]
    return sorted(files, key=natural_key)


selection = []   # liste de (chemin, étiquette, numéro de classe, id de l'image originale)
for class_num, label in enumerate(SELECTED_LABELS):
    files = collect_images(label)
    chosen = random.sample(files, min(N_PER_SYMBOL, len(files)))
    print(f"{label:<10} : {len(files):>4} images disponibles -> {len(chosen)} retenues")
    for f in sorted(chosen, key=natural_key):
        selection.append((f, label, class_num, len(selection)))

print(f"\nTotal d'images originales retenues : {len(selection)}")

In [ ]:
# Vérification visuelle des originaux sélectionnés (écarter les mauvaises images via EXCLUDE_FILES puis relancer)
originals = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p, _, _, _ in selection]
print("Tailles originales rencontrées (largeur x hauteur) :",
      Counter((im.shape[1], im.shape[0]) for im in originals if im is not None))
show_grid(originals, [f"{p.name}" for p, _, _, _ in selection], ncols=8, size=2.2)

---
## 2 | PRÉTRAITEMENT DES IMAGES
1. **Niveaux de gris** (lecture avec `cv2.IMREAD_GRAYSCALE`) : un canal au lieu de trois, moins de caractéristiques.
2. **Réduction de taille** par `RESIZE_FACTOR` (`INTER_AREA`, recommandé pour réduire).
3. **Extraction automatique de la ROI** avec OpenCV :
   flou gaussien → seuillage d'Otsu → fond mis à noir (polarité détectée sur les bords) → fermeture morphologique → contours externes → boîte englobante du plus grand contour (+ marge).
   Si aucun contour valable n'est trouvé, on conserve l'image entière (et on le signale).
4. **Redimensionnement de la ROI** à `ROI_SIZE` : toutes les images ont exactement le même nombre de pixels.

In [ ]:
def resize_image(img, factor):
    if factor is None or factor == 1:
        return img
    h, w = img.shape[:2]
    return cv2.resize(img, (max(1, int(w * factor)), max(1, int(h * factor))), interpolation=cv2.INTER_AREA)


def binarize(gray):
    """Image binaire : forme en blanc (255), fond en noir (0)."""
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, th = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Polarité : si les bords sont majoritairement blancs, c'est que le fond est blanc -> on inverse
    border = np.concatenate([th[0, :], th[-1, :], th[:, 0], th[:, -1]])
    if border.mean() > 127:
        th = cv2.bitwise_not(th)
    return th


def extract_roi(gray):
    """Retourne (roi, boîte (x0, y0, x1, y1), succès)."""
    h, w = gray.shape
    th = binarize(gray)
    th = cv2.morphologyEx(th, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    cnts = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cnts = imutils.grab_contours(cnts)
    if cnts:
        c = max(cnts, key=cv2.contourArea)
        if cv2.contourArea(c) >= MIN_ROI_AREA_RATIO * h * w:
            x, y, bw, bh = cv2.boundingRect(c)
            mx, my = int(bw * ROI_MARGIN), int(bh * ROI_MARGIN)
            x0, y0 = max(0, x - mx), max(0, y - my)
            x1, y1 = min(w, x + bw + mx), min(h, y + bh + my)
            return gray[y0:y1, x0:x1], (x0, y0, x1, y1), True
    return gray, (0, 0, w, h), False

In [ ]:
# base_records : une entrée par image ORIGINALE prétraitée
base_records = []
debug_imgs, debug_titles = [], []
n_fail = 0

for (path, label, class_num, group), gray in zip(selection, originals):
    if gray is None:
        print(f"[!] Lecture impossible : {path}")
        continue
    small = resize_image(gray, RESIZE_FACTOR)
    roi, (x0, y0, x1, y1), ok = extract_roi(small)
    if not ok:
        n_fail += 1
        print(f"[!] ROI non trouvée pour {path.name} -> image entière conservée")
    roi = cv2.resize(roi, ROI_SIZE, interpolation=cv2.INTER_AREA)

    base_records.append({"img": roi, "label": label, "class_num": class_num,
                         "group": group, "is_aug": False, "name": path.stem})

    dbg = small.copy()
    cv2.rectangle(dbg, (x0, y0), (x1 - 1, y1 - 1), 255, 1)
    debug_imgs += [dbg, roi]
    debug_titles += [f"{label} (détection)", f"{label} (ROI)"]

print(f"\n{len(base_records)} images prétraitées, {n_fail} ROI en échec.")
show_grid(debug_imgs[:32], debug_titles[:32], ncols=8, size=2.2)   # contrôle visuel des 16 premières

---
## 3 | ALTÉRATIONS D'IMAGES
**Altérations appliquées (à reprendre dans le rapport) :**
- Niveaux de gris (toutes les images).
- Flou gaussien : `APPLY_BLUR` (voir paramètres).
- **Augmentation de données** (appliquée sur la ROI, après la sélection), 7 opérations tirées au hasard, chacune avec une probabilité de 50 % :
  rotation (±15°), translation (±8 %), zoom (0,9 à 1,1), déformation perspective, luminosité/contraste, bruit gaussien, flou gaussien ; retournements optionnels (`USE_FLIPS`).
- Chaque image augmentée hérite du `group` de son image d'origine : **à utiliser pour séparer entraînement/test sans fuite de données** (`GroupKFold` / découpage par groupe).

In [ ]:
def _affine(img, M):
    return cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), borderMode=cv2.BORDER_REPLICATE)


def aug_rotation(img, rng, max_deg=15):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), rng.uniform(-max_deg, max_deg), 1.0)
    return _affine(img, M)


def aug_translation(img, rng, max_frac=0.08):
    h, w = img.shape
    M = np.float32([[1, 0, rng.uniform(-max_frac, max_frac) * w],
                    [0, 1, rng.uniform(-max_frac, max_frac) * h]])
    return _affine(img, M)


def aug_zoom(img, rng, lo=0.9, hi=1.1):
    h, w = img.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), 0, rng.uniform(lo, hi))
    return _affine(img, M)


def aug_perspective(img, rng, mag=0.07):
    h, w = img.shape
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    dst = src + (rng.uniform(-mag, mag, (4, 2)) * np.float32([w, h])).astype(np.float32)
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (w, h), borderMode=cv2.BORDER_REPLICATE)


def aug_brightness_contrast(img, rng):
    return cv2.convertScaleAbs(img, alpha=rng.uniform(0.7, 1.3), beta=rng.uniform(-30, 30))


def aug_noise(img, rng, sigma_max=12):
    noisy = img.astype(np.float32) + rng.normal(0, rng.uniform(2, sigma_max), img.shape)
    return np.clip(noisy, 0, 255).astype(np.uint8)


def aug_blur(img, rng):
    k = int(rng.choice([3, 5]))
    return cv2.GaussianBlur(img, (k, k), 0)


def aug_flip(img, rng):
    return cv2.flip(img, int(rng.choice([0, 1, -1])))


AUG_OPS = [aug_rotation, aug_translation, aug_zoom, aug_perspective,
           aug_brightness_contrast, aug_noise, aug_blur]
if USE_FLIPS:
    AUG_OPS.append(aug_flip)


def augment(img, rng, p=0.5):
    """Applique chaque opération avec probabilité p (au moins une opération est toujours appliquée)."""
    out = img.copy()
    applied = [op for op in AUG_OPS if rng.random() < p]
    if not applied:
        applied = [AUG_OPS[rng.integers(len(AUG_OPS))]]
    for op in applied:
        out = op(out, rng)
    return out

In [ ]:
# Flou optionnel sur les images de base
if APPLY_BLUR:
    for r in base_records:
        r["img"] = cv2.GaussianBlur(r["img"], BLUR_KERNEL, 0)

# Augmentation
aug_records = []
for r in base_records:
    for k in range(N_AUG_PER_IMAGE):
        aug_records.append({"img": augment(r["img"], rng), "label": r["label"], "class_num": r["class_num"],
                            "group": r["group"], "is_aug": True, "name": f"aug{k}_{r['name']}"})

assert len(aug_records) >= 100, "L'énoncé exige au moins 100 images supplémentaires : augmentez N_AUG_PER_IMAGE."
all_records = base_records + aug_records
print(f"{len(base_records)} originales + {len(aug_records)} augmentées = {len(all_records)} images")

# Aperçu : une image originale et 15 versions augmentées
ex = base_records[0]
preview = [ex["img"]] + [augment(ex["img"], rng) for _ in range(15)]
show_grid(preview, ["original"] + [f"aug {i+1}" for i in range(15)], ncols=8, size=1.8)

---
## 4 | EXTRACTION DES CARACTÉRISTIQUES
- **Base :** les pixels de la ROI (`ROI_SIZE`), aplatis en vecteur et divisés par 255 (valeurs dans [0, 1]).
- **Optionnel (`USE_EXTRA_FEATURES`) :**
  - *nombre de formes* : nombre de contours (hiérarchie complète, aire minimale pour ignorer le bruit) sur l'image binarisée, divisé par `MAX_SHAPES` ;
  - *excentricité de la plus grande forme* : calculée avec les moments du contour, `sqrt(1 - λ2/λ1)` (0 = cercle, → 1 = très allongé).
- **Normalisation :** ces valeurs sont déjà dans [0, 1] par construction (constantes fixes, aucune statistique du jeu de données => pas de fuite).
  Pour la suite du labo, ajustez un `MinMaxScaler` / `StandardScaler` **sur l'entraînement uniquement**.

In [ ]:
def shape_features(roi):
    """(nombre de formes, excentricité de la plus grande forme) à partir d'une ROI en niveaux de gris."""
    th = binarize(roi)
    cnts = imutils.grab_contours(cv2.findContours(th, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE))
    min_area = 0.002 * roi.shape[0] * roi.shape[1]
    cnts = [c for c in cnts if cv2.contourArea(c) >= min_area]
    if not cnts:
        return 0, 0.0
    c = max(cnts, key=cv2.contourArea)
    m = cv2.moments(c)
    if m["m00"] == 0:
        return len(cnts), 0.0
    mu20, mu02, mu11 = m["mu20"] / m["m00"], m["mu02"] / m["m00"], m["mu11"] / m["m00"]
    common = np.sqrt(4 * mu11 ** 2 + (mu20 - mu02) ** 2)
    l1, l2 = (mu20 + mu02 + common) / 2, (mu20 + mu02 - common) / 2
    ecc = float(np.sqrt(max(0.0, 1 - l2 / l1))) if l1 > 0 else 0.0
    return len(cnts), ecc


def extract_features(roi):
    pixels = roi.astype(np.float32).ravel() / 255.0
    if not USE_EXTRA_FEATURES:
        return pixels
    n_shapes, ecc = shape_features(roi)
    return np.concatenate([pixels, [min(n_shapes, MAX_SHAPES) / MAX_SHAPES, ecc]]).astype(np.float32)

In [ ]:
# Mélange, puis construction de X, y (+ métadonnées pour un découpage sans fuite)
order = rng.permutation(len(all_records))
shuffled = [all_records[i] for i in order]

X = np.stack([extract_features(r["img"]) for r in shuffled])
y = [r["label"] for r in shuffled]
groups = np.array([r["group"] for r in shuffled])      # id de l'image originale (augmentées incluses)
is_aug = np.array([r["is_aug"] for r in shuffled])

print("X :", X.shape, "| y :", len(y), "| caractéristiques par image :", X.shape[1],
      f"({ROI_SIZE[0] * ROI_SIZE[1]} pixels" + (" + 2 extras)" if USE_EXTRA_FEATURES else ")"))
print("Répartition des classes :", dict(sorted(Counter(y).items())))

In [ ]:
# Sauvegarde des images prétraitées, classées par étiquette (à inclure dans le .zip)
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
for r in all_records:
    d = OUT_DIR / r["label"]
    d.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(d / f"{r['name']}_{r['label']}.png"), r["img"])

# Sauvegarde des données (même format que le tutoriel + métadonnées)
for name, obj in [("X", X), ("y", y), ("groups", groups), ("is_aug", is_aug),
                  ("meta", {"labels": SELECTED_LABELS, "roi_size": ROI_SIZE,
                            "use_extra_features": USE_EXTRA_FEATURES, "max_shapes": MAX_SHAPES})]:
    with open(f"{name}.pickle", "wb") as f:
        pickle.dump(obj, f)

print(f"Images enregistrées dans : {OUT_DIR}")
print("Fichiers : X.pickle, y.pickle, groups.pickle, is_aug.pickle, meta.pickle")

In [ ]:
# Contrôle final : rechargement + affichage de 16 images aléatoires reconstruites depuis X
with open("X.pickle", "rb") as f:
    X_chk = pickle.load(f)
with open("y.pickle", "rb") as f:
    y_chk = pickle.load(f)
assert X_chk.shape == X.shape and len(y_chk) == len(X_chk)

W, H = ROI_SIZE
idx = random.sample(range(len(X_chk)), 16)
show_grid([(X_chk[i][:W * H].reshape(H, W) * 255).astype(np.uint8) for i in idx],
          [str(y_chk[i]) for i in idx], ncols=8, size=2.0)
if USE_EXTRA_FEATURES:
    print("Exemple de caractéristiques supplémentaires (nb formes normalisé, excentricité) :", X_chk[idx[0]][-2:])